# Q-ERA Stage-by-Stage Runner

This notebook is for team members running the frozen v1.1 implementation. It invokes tested `qera` package code rather than duplicating formulas.

Prerequisites:

- Select the workspace `.venv-classiq` interpreter.
- Authenticate once before a Classiq platform cell.
- Set `RUN_CLASSIQ = True` only when you intend to create new remote jobs.

It reproduces Stages 0–5. Stage 6 is intentionally paused.


## Outline

1. Environment and tests
2. Exact classical truth
3. QUBO/Ising/aligned-energy proof
4. Classiq model, synthesis, execution, and decoding
5. Static and random controls
6. Three-solve adaptive experiment


In [ ]:
from __future__ import annotations

import importlib.metadata as metadata
import json
import subprocess
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
IMPLEMENTATION_ROOT = next(p for p in (cwd, cwd / 'implementation', cwd.parent) if (p / 'qera').exists())
sys.path.insert(0, str(IMPLEMENTATION_ROOT))
PYTHON = Path(sys.executable)
RUN_CLASSIQ = False  # Opt in to synthesis and simulator jobs.

def run_script(name: str, *args: object) -> None:
    command = [str(PYTHON), str(IMPLEMENTATION_ROOT / 'scripts' / name), *map(str, args)]
    print('>', ' '.join(command))
    subprocess.run(command, cwd=IMPLEMENTATION_ROOT, check=True)

print('Implementation root:', IMPLEMENTATION_ROOT)
print('Python:', PYTHON)
print('Classiq jobs enabled:', RUN_CLASSIQ)


## Stage 0 — environment

Verify the actual kernel and dependencies. Authentication is a separate opt-in action.


In [ ]:
print('Python:', sys.version.split()[0])
for package in ('classiq', 'numpy', 'pandas', 'matplotlib', 'networkx', 'pytest'):
    print(f'{package}: {metadata.version(package)}')
AUTHENTICATE = False
if AUTHENTICATE:
    import classiq
    classiq.authenticate(overwrite=True)


## Stage 1 — exact classical truth

Regenerate feasibility counts, optimum tie sets, and exact adaptive trajectories.


In [ ]:
run_script('generate_exact_truth.py')
exact_truth = json.loads((IMPLEMENTATION_ROOT / 'artifacts/tables/exact_truth.json').read_text())
print('Assignments:', exact_truth['assignment_count'])
print('Joint feasible:', exact_truth['joint_feasible_count'])
print('Minimax ties:', exact_truth['pure_minimax_regret']['assignments'])
print('Findings:', exact_truth['implementation_findings'])


## Stage 2 — exhaustive encoding proof

Check direct/QUBO agreement on 81 routes and QUBO/Ising agreement on all 4096 bitstrings for all six adaptive energies.


In [ ]:
run_script('generate_encoding_proof.py')
encoding = json.loads((IMPLEMENTATION_ROOT / 'artifacts/tables/encoding_proof.json').read_text())
print('Proof runs:', len(encoding['runs']))
print('M:', sorted({r['M'] for r in encoding['runs']}))
print('Lambda:', sorted({r['Lambda'] for r in encoding['runs']}))
print('Max QUBO error:', max(r['maximum_valid_qubo_error'] for r in encoding['runs']))
print('Max Ising error:', max(r['maximum_ising_error'] for r in encoding['runs']))
print('Aligned grounds feasible:', all(r['all_aligned_ground_states_joint_feasible'] for r in encoding['runs']))


## Stage 3 — one working Classiq solve

Serialize locally first. The guarded cell then synthesizes, runs 10 optimizer iterations at 512 shots, takes 4096 final shots, and decodes the named `routes` output.


In [ ]:
run_script('create_qaoa_qmod.py')
if RUN_CLASSIQ:
    run_script('synthesize_qaoa.py', '--objective-mode', 'cost')
    run_script('execute_qaoa_smoke.py', '--objective-mode', 'cost', '--max-iteration', 10)
    run_script('process_qaoa_smoke.py', '--run-name', 'uniform_cost_p1_smoke')
else:
    print('Using saved Stage 3 artifacts.')
stage3 = json.loads((IMPLEMENTATION_ROOT / 'artifacts/runs/uniform_cost_p1_smoke/summary.json').read_text())
stage3


## Stage 4 — fair static controls

Generate exact, heuristic, and random controls. Optionally rerun the matched uniform-regret QAOA control.


In [ ]:
run_script('generate_static_controls.py')
if RUN_CLASSIQ:
    run_script('synthesize_qaoa.py', '--objective-mode', 'regret')
    run_script('execute_qaoa_smoke.py', '--objective-mode', 'regret', '--max-iteration', 10)
    run_script('process_qaoa_smoke.py', '--run-name', 'uniform_regret_p1_smoke')
else:
    print('Using saved uniform-regret artifacts.')
controls = json.loads((IMPLEMENTATION_ROOT / 'artifacts/tables/static_controls.json').read_text())
regret_qaoa = json.loads((IMPLEMENTATION_ROOT / 'artifacts/runs/uniform_regret_p1_smoke/summary.json').read_text())
print('Shortest path:', controls['controls']['shortest_path'])
print('Load-aware greedy:', controls['controls']['load_aware_greedy_all_training_scenarios'])
print('Uniform-regret QAOA:', regret_qaoa)


## Stage 5 — adaptive exact and QAOA runs

Prove the shared exact loop, then optionally reproduce QAOA iterations 1 and 2 from the Stage 3 result using scale-corrected gamma warm starts.


In [ ]:
from qera.adaptive import run_adaptive
from qera.evaluate import Evaluator
from qera.exact import ExactInnerSolver

evaluator = Evaluator()
exact_adaptive = run_adaptive(ExactInnerSolver(evaluator), evaluator, 'cost')
[(s.iteration, s.request.scenario_weights, s.result.assignment, s.worst_regret) for s in exact_adaptive.steps]


In [ ]:
if RUN_CLASSIQ:
    run_script('prepare_adaptive_next.py', '--previous-run', 'uniform_cost_p1_smoke', '--next-name', 'adaptive_cost_t1')
    p1 = json.loads((IMPLEMENTATION_ROOT / 'artifacts/runs/adaptive_cost_t1.prepared.json').read_text())
    run_script('synthesize_qaoa.py', '--objective-mode', 'cost', '--weights', *p1['weights'], '--artifact-name', 'adaptive_cost_t1')
    run_script('execute_qaoa_smoke.py', '--objective-mode', 'cost', '--weights', *p1['weights'], '--qprog', 'artifacts/circuits/adaptive_cost_t1.qprog', '--run-name', 'adaptive_cost_t1', '--initial-params', *p1['warm_start_parameters'], '--max-iteration', 10)
    run_script('process_qaoa_smoke.py', '--run-name', 'adaptive_cost_t1')
else:
    print('Using saved adaptive iteration 1 artifacts.')


In [ ]:
if RUN_CLASSIQ:
    run_script('prepare_adaptive_next.py', '--previous-run', 'adaptive_cost_t1', '--next-name', 'adaptive_cost_t2')
    p2 = json.loads((IMPLEMENTATION_ROOT / 'artifacts/runs/adaptive_cost_t2.prepared.json').read_text())
    run_script('synthesize_qaoa.py', '--objective-mode', 'cost', '--weights', *p2['weights'], '--artifact-name', 'adaptive_cost_t2')
    run_script('execute_qaoa_smoke.py', '--objective-mode', 'cost', '--weights', *p2['weights'], '--qprog', 'artifacts/circuits/adaptive_cost_t2.qprog', '--run-name', 'adaptive_cost_t2', '--initial-params', *p2['warm_start_parameters'], '--max-iteration', 10)
    run_script('process_qaoa_smoke.py', '--run-name', 'adaptive_cost_t2')
else:
    print('Using saved adaptive iteration 2 artifacts.')
run_script('summarize_adaptive_qaoa.py')
adaptive = json.loads((IMPLEMENTATION_ROOT / 'artifacts/runs/adaptive_cost/summary.json').read_text())
print('Status:', adaptive['status'])
print('Best assignment:', adaptive['best_assignment'])
[(s['iteration'], s['result']['assignment'], s['worst_regret']) for s in adaptive['steps']]


## Validation, pitfalls, and exercise

- Never duplicate evaluator formulas in this notebook.
- Decode the named `routes` output, not provider bitstring order.
- Scaling changes gamma, not depth.
- `RUN_CLASSIQ=True` creates remote jobs and may consume quota.

Exercise: compare the three saved runs. Which has the largest joint-feasible probability, and is it necessarily the one with the best worst-case regret?


In [ ]:
run_names = ('uniform_cost_p1_smoke', 'adaptive_cost_t1', 'adaptive_cost_t2')
comparison = {}
for name in run_names:
    summary = json.loads((IMPLEMENTATION_ROOT / 'artifacts/runs' / name / 'summary.json').read_text())
    comparison[name] = {
        'joint_feasible_probability': summary['joint_feasible_probability'],
        'selected_assignment': summary['selected_assignment'],
    }
comparison


In [ ]:
subprocess.run([str(PYTHON), '-m', 'pytest', str(IMPLEMENTATION_ROOT / 'tests')], cwd=IMPLEMENTATION_ROOT, check=True)
